# 00 — Setup and local health

Locate the project, load the pilot configuration, and inspect the local environment. This is the first notebook. The optional Ollama check lists installed models only and never performs inference or pulls a model.

In [ ]:
import importlib.metadata as metadata
import json
import platform
import sys
from pathlib import Path

try:
    from rubric_order.api import (
        check_ollama,
        find_project_root,
        load_experiment_config,
        resolved_configuration,
    )
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "Install the project editable in this kernel environment: "
        "python -m pip install --no-deps -e ."
    ) from exc

PROJECT_ROOT = find_project_root(Path.cwd())
PROJECT_ROOT

In [ ]:
# Edit this cell for another dataset, configuration, or run.
DATA_PATH = PROJECT_ROOT / "HelpSteer2"
CONFIG_PATH = PROJECT_ROOT / "configs" / "pilot.yaml"
RUN_ID = "pilot-001"
RUNTIME_OVERRIDES = {}

In [ ]:
config = load_experiment_config(CONFIG_PATH, DATA_PATH, overrides=RUNTIME_OVERRIDES)
print(json.dumps(resolved_configuration(config), indent=2))

In [ ]:
packages = [
    "rubric-order",
    "PyYAML",
    "pyarrow",
    "datasets",
    "matplotlib",
    "pandas",
    "nbformat",
    "nbclient",
]
versions = {}
for package in packages:
    try:
        versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        versions[package] = "not installed"
{
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "executable": sys.executable,
    "packages": versions,
}

In [ ]:
# Leave False for a fully offline setup pass. This check performs no inference.
CHECK_OLLAMA = False

In [ ]:
if CHECK_OLLAMA:
    try:
        health = check_ollama(config, stage="both")
    except RuntimeError as exc:
        raise RuntimeError(
            "Ollama is unavailable. Start the local server and rerun this cell."
        ) from exc
    if health["missing"]:
        print("Missing local Ollama models:", ", ".join(health["missing"]))
        print("Install them manually before the canary; this notebook will not pull models.")
    else:
        print("Ollama is reachable and both configured models are installed.")
    display(health)
else:
    print("Ollama check skipped. Set CHECK_OLLAMA = True to list local models.")